In [22]:
import json
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         11,
    "axes.titlesize":    12,
    "figure.dpi":        150,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

ACOUSTIC_JSON_PATH = "new_bert_ac_dict.json"
PARQUET_PATH       = r"C:\Users\super\Desktop\Data Thesis\final_df_clustered.parquet"

ACOUSTIC_FEATURE_NAMES = [
    "pct_vol_high", "pct_vol_low",
    "pct_pitch_high", "pct_pitch_low",
    "std_volume", "std_pitch"
]

print("Loading acoustic summary features...")
with open(ACOUSTIC_JSON_PATH, "r") as f:
    raw_data = json.load(f)

records = []
for clip_key, entry in raw_data.items():
    acoustics = entry.get("acoustics", [])
    if len(acoustics) == 6:
        vals = [round(float(v), 6) for v in acoustics]
        rec = {"clip_key": clip_key}
        rec.update(dict(zip(ACOUSTIC_FEATURE_NAMES, vals)))
        records.append(rec)

acoustic_df = pd.DataFrame(records)

vector_counts = acoustic_df[ACOUSTIC_FEATURE_NAMES].value_counts()
placeholder_vector = vector_counts.index[0]
acoustic_df["is_placeholder"] = (
    acoustic_df[ACOUSTIC_FEATURE_NAMES].apply(tuple, axis=1) == placeholder_vector
)

n_placeholder = acoustic_df["is_placeholder"].sum()
print(f"Flagged {n_placeholder} / {len(acoustic_df)} clips "
      f"({n_placeholder/len(acoustic_df)*100:.1f}%) as placeholder (no detected speech).")

acoustic_df_real = acoustic_df[~acoustic_df["is_placeholder"]].drop(columns="is_placeholder")
print(f"Remaining clips with genuine acoustic signal: {len(acoustic_df_real)}")

Loading acoustic summary features...
Flagged 9414 / 12197 clips (77.2%) as placeholder (no detected speech).
Remaining clips with genuine acoustic signal: 2783


In [23]:
final_df = pd.read_parquet(PARQUET_PATH)

clip_meta = (
    final_df.groupby("clip_id")
    .agg(behavioral_cluster=("behavioral_cluster", "first"),
         engagement=("engagement", "first"),
         participant_id=("participant_id", "first"))
    .reset_index()
)

print(f"Total clips in your clustered data: {len(clip_meta)}")

merged = clip_meta.merge(acoustic_df_real, left_on="clip_id", right_on="clip_key", how="inner")
print(f"Matched clips with GENUINE acoustic signal: {len(merged)} / {len(clip_meta)} "
      f"({len(merged)/len(clip_meta)*100:.1f}%)")

print("\nMatched clips by cluster:")
print(merged.groupby("behavioral_cluster").size())

Total clips in your clustered data: 12093
Matched clips with GENUINE acoustic signal: 2761 / 12093 (22.8%)

Matched clips by cluster:
behavioral_cluster
0     938
1    1823
dtype: int64


In [24]:
all_acoustic = clip_meta.merge(
    acoustic_df[["clip_key", "is_placeholder"]],
    left_on="clip_id", right_on="clip_key", how="left"
)
all_acoustic["has_speech"] = ~all_acoustic["is_placeholder"].fillna(True)

coverage_by_cluster = (
    all_acoustic.groupby("behavioral_cluster")["has_speech"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "n_with_speech", "count": "n_total", "mean": "speech_rate"})
)
print("Proportion of clips with detected speech, by cluster:")
print(coverage_by_cluster)

from scipy.stats import chi2_contingency
ctab = pd.crosstab(all_acoustic["behavioral_cluster"], all_acoustic["has_speech"])
chi2, p_val, dof, expected = chi2_contingency(ctab)
print(f"\nChi-square test: chi2={chi2:.3f}, p={p_val:.6f}")
print("If p < 0.05: clusters differ significantly in HOW OFTEN they speak at all —")
print("this is itself a meaningful, independent behavioral finding worth reporting.")

Proportion of clips with detected speech, by cluster:
                    n_with_speech  n_total  speech_rate
behavioral_cluster                                     
0                             938     3512     0.267084
1                            1823     8581     0.212446

Chi-square test: chi2=41.916, p=0.000000
If p < 0.05: clusters differ significantly in HOW OFTEN they speak at all —
this is itself a meaningful, independent behavioral finding worth reporting.


In [25]:
results = []
for feature in ACOUSTIC_FEATURE_NAMES:
    c0_vals = merged.loc[merged["behavioral_cluster"] == 0, feature].dropna()
    c1_vals = merged.loc[merged["behavioral_cluster"] == 1, feature].dropna()

    t_stat, p_val = scipy_stats.ttest_ind(c0_vals, c1_vals, equal_var=False)
    pooled_std = np.sqrt((c0_vals.std()**2 + c1_vals.std()**2) / 2)
    cohens_d = (c0_vals.mean() - c1_vals.mean()) / pooled_std if pooled_std > 0 else np.nan

    results.append({
        "feature": feature,
        "cluster0_mean": c0_vals.mean(), "cluster0_std": c0_vals.std(),
        "cluster1_mean": c1_vals.mean(), "cluster1_std": c1_vals.std(),
        "t_stat": t_stat, "p_value": p_val, "cohens_d": cohens_d,
        "n_cluster0": len(c0_vals), "n_cluster1": len(c1_vals),
    })

results_df = pd.DataFrame(results)
print("Acoustic feature comparison: Cluster 0 vs Cluster 1 (genuine speech only)")
print(results_df.round(4).to_string(index=False))

n_significant = (results_df["p_value"] < 0.05).sum()
print(f"\n{n_significant} of {len(ACOUSTIC_FEATURE_NAMES)} features differ significantly (p < 0.05).")

Acoustic feature comparison: Cluster 0 vs Cluster 1 (genuine speech only)
       feature  cluster0_mean  cluster0_std  cluster1_mean  cluster1_std  t_stat  p_value  cohens_d  n_cluster0  n_cluster1
  pct_vol_high         0.0271        1.0002        -0.0298        0.9383  1.4464   0.1483    0.0587         938        1823
   pct_vol_low         0.0875        0.9461        -0.0364        1.0281  3.1616   0.0016    0.1253         938        1823
pct_pitch_high         0.0561        1.0665        -0.0312        0.9798  2.0918   0.0366    0.0852         938        1823
 pct_pitch_low        -0.0784        0.9603         0.0586        1.0240 -3.4700   0.0005   -0.1380         938        1823
    std_volume        -0.0874        0.9461         0.0363        1.0280 -3.1580   0.0016   -0.1252         938        1823
     std_pitch        -0.0887        0.9981         0.0666        0.9920 -3.8791   0.0001   -0.1560         938        1823

5 of 6 features differ significantly (p < 0.05).
